In [2]:
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
import tensorflow as tf


# -----------------------------
# 1. 기본 유틸
# -----------------------------
def load_all_records_from_tfrecord(tfrecord_path, compression_type=""):
    """TFRecord 파일의 raw record(bytes)들을 리스트로 읽음."""
    return list(tf.data.TFRecordDataset(tfrecord_path, compression_type=compression_type))


def parse_example_from_record(record):
    """raw record(bytes or tf.Tensor) -> tf.train.Example"""
    example = tf.train.Example()
    if hasattr(record, "numpy"):
        example.ParseFromString(record.numpy())
    else:
        example.ParseFromString(record)
    return example


def get_feature_array(example, name, dtype):
    feat = example.features.feature[name]

    if dtype == np.float32:
        return np.array(feat.float_list.value, dtype=np.float32)
    elif dtype == np.int64:
        return np.array(feat.int64_list.value, dtype=np.int64)
    else:
        raise ValueError(f"Unsupported dtype: {dtype}")


def set_feature_array(example, name, arr):
    feat = example.features.feature[name]
    arr = np.asarray(arr)

    del feat.float_list.value[:]
    del feat.int64_list.value[:]
    del feat.bytes_list.value[:]

    if np.issubdtype(arr.dtype, np.floating):
        feat.float_list.value.extend(arr.astype(np.float32).reshape(-1).tolist())
    elif np.issubdtype(arr.dtype, np.integer):
        feat.int64_list.value.extend(arr.astype(np.int64).reshape(-1).tolist())
    else:
        raise ValueError(f"Unsupported dtype for feature {name}: {arr.dtype}")


def infer_num_objects(example):
    """
    tf.Example에서 object 수 추정.
    state/is_sdc는 [num_objects] shape의 int64 feature.
    """
    return len(example.features.feature["state/is_sdc"].int64_list.value)


def reshape_feature(example, name, second_dim, dtype):
    """
    1D flat feature를 [num_objects, second_dim]으로 reshape
    """
    num_objects = infer_num_objects(example)
    flat = get_feature_array(example, name, dtype=dtype)
    expected = num_objects * second_dim
    if len(flat) != expected:
        raise ValueError(
            f"Feature {name} has length {len(flat)}, "
            f"expected {expected} = {num_objects} x {second_dim}"
        )
    return flat.reshape(num_objects, second_dim)


def find_sdc_track_index(example):
    """
    tf.Example에서 ego(SDC) row index 찾기.
    state/is_sdc == 1 인 row를 찾음.
    """
    is_sdc = get_feature_array(example, "state/is_sdc", np.int64)
    ego_indices = np.where(is_sdc == 1)[0]

    if len(ego_indices) != 1:
        raise ValueError(f"Expected exactly one SDC row, got {len(ego_indices)}")

    return int(ego_indices[0])


def overwrite_ego_track_states(example, ego_traj):
    """
    ego_traj: (T, 5) [x, y, yaw, vel_x, vel_y]

    Waymo tf.Example motion format은 일반적으로
      past   : 10 step
      current: 1 step
      future : 80 step
    총 91 timestep.

    기존 코드와 동일하게:
      - x, y, heading, velocity_x, velocity_y 만 교체
      - z, length, width, height 는 원본 유지
      - valid는 1로 설정
    """
    ego_idx = find_sdc_track_index(example)

    num_timesteps = 91
    if ego_traj.shape[0] != num_timesteps:
        raise ValueError(
            f"Trajectory timestep mismatch: ego_traj has {ego_traj.shape[0]} "
            f"but tf.Example format expects {num_timesteps}"
        )

    # 분할
    past_traj = ego_traj[:10]       # [10, 5]
    current_traj = ego_traj[10:11]  # [1, 5]
    future_traj = ego_traj[11:]     # [80, 5]

    # 수정 대상 feature 목록
    segments = [
        ("past", 10, past_traj),
        ("current", 1, current_traj),
        ("future", 80, future_traj),
    ]

    for segment_name, seg_len, seg_traj in segments:
        # 원본 feature 읽기
        x = reshape_feature(example, f"state/{segment_name}/x", seg_len, np.float32)
        y = reshape_feature(example, f"state/{segment_name}/y", seg_len, np.float32)
        bbox_yaw = reshape_feature(example, f"state/{segment_name}/bbox_yaw", seg_len, np.float32)
        velocity_x = reshape_feature(example, f"state/{segment_name}/velocity_x", seg_len, np.float32)
        velocity_y = reshape_feature(example, f"state/{segment_name}/velocity_y", seg_len, np.float32)

        # valid는 int64인 경우가 많음
        valid = reshape_feature(example, f"state/{segment_name}/valid", seg_len, np.int64)

        # ego row만 덮어쓰기
        x[ego_idx] = seg_traj[:, 0]
        y[ego_idx] = seg_traj[:, 1]
        bbox_yaw[ego_idx] = seg_traj[:, 2]
        velocity_x[ego_idx] = seg_traj[:, 3]
        velocity_y[ego_idx] = seg_traj[:, 4]
        valid[ego_idx] = 1

        # 다시 저장
        set_feature_array(example, f"state/{segment_name}/x", x)
        set_feature_array(example, f"state/{segment_name}/y", y)
        set_feature_array(example, f"state/{segment_name}/bbox_yaw", bbox_yaw)
        set_feature_array(example, f"state/{segment_name}/velocity_x", velocity_x)
        set_feature_array(example, f"state/{segment_name}/velocity_y", velocity_y)
        set_feature_array(example, f"state/{segment_name}/valid", valid)

    return example


def maybe_append_scenario_id_suffix(example, scenario_id_suffix="_modified"):
    """
    tf.Example 안에 scenario/id feature가 있으면 suffix를 붙임.
    없으면 그냥 그대로 둠.
    """
    if not scenario_id_suffix:
        return example

    key = "scenario/id"
    if key not in example.features.feature:
        return example

    feat = example.features.feature[key]
    if len(feat.bytes_list.value) == 0:
        return example

    original = feat.bytes_list.value[0].decode("utf-8")
    new_value = (original + scenario_id_suffix).encode("utf-8")

    del feat.bytes_list.value[:]
    feat.bytes_list.value.append(new_value)
    return example


def parse_npz_filename(npz_path):
    """
    기존 네 코드의 filename parsing 로직 유지
    """
    stem = Path(npz_path).stem
    tfrecord_name = ".".join(stem.split(".")[:-1])
    scenario_index = int(stem.split(".")[-1].split("_")[-1])
    return tfrecord_name, scenario_index


def load_ego_trajectory_from_npz(npz_path, key="trajectories_world_bkt5"):
    """
    기존 네 코드 로직 유지:
    npz[key][0] -> (T, 5)
    """
    data = np.load(npz_path)
    ego_traj = data[key][0]
    return ego_traj


# -----------------------------
# 3. 메인 로직
# -----------------------------
def build_modified_tfrecord(
    source_tfrecord_dir,
    npz_paths,
    output_tfrecord_path,
    npz_key="trajectories_world_bkt5",
    scenario_id_suffix="_modified",
    compression_type="",
):
    """
    source_tfrecord_dir:
      원본 waymo tfrecord 파일들이 있는 디렉토리

    npz_paths:
      {tfrecord_name}_{scenario_index}.npz 파일들의 경로 리스트

    output_tfrecord_path:
      수정된 scenario들만 모아 저장할 새 tfrecord 경로
    """

    # tfrecord별로 어떤 scenario index들을 수정해야 하는지 정리
    jobs_by_tfrecord = defaultdict(list)
    for npz_path in npz_paths:
        tfrecord_name, scenario_index = parse_npz_filename(npz_path)
        jobs_by_tfrecord[tfrecord_name].append((scenario_index, npz_path))

    # index 순으로 처리
    for tfrecord_name in jobs_by_tfrecord:
        jobs_by_tfrecord[tfrecord_name].sort(key=lambda x: x[0])

    num_written = 0

    with tf.io.TFRecordWriter(output_tfrecord_path) as writer:
        for tfrecord_name, jobs in sorted(jobs_by_tfrecord.items()):
            tfrecord_path = os.path.join(source_tfrecord_dir, tfrecord_name)
            if not os.path.exists(tfrecord_path):
                raise FileNotFoundError(f"Source tfrecord not found: {tfrecord_path}")

            records = load_all_records_from_tfrecord(
                tfrecord_path,
                compression_type=compression_type,
            )

            for scenario_index, npz_path in jobs:
                if scenario_index < 0 or scenario_index >= len(records):
                    raise IndexError(
                        f"scenario_index={scenario_index} out of range for {tfrecord_name} "
                        f"(num_scenarios={len(records)})"
                    )

                example = parse_example_from_record(records[scenario_index])

                ego_traj = load_ego_trajectory_from_npz(npz_path, key=npz_key)

                example = overwrite_ego_track_states(example, ego_traj)
                example = maybe_append_scenario_id_suffix(example, scenario_id_suffix)

                writer.write(example.SerializeToString())
                num_written += 1

                print(
                    f"[OK] wrote modified example from "
                    f"{tfrecord_name}[{scenario_index}] "
                    f"-> {Path(npz_path).name}"
                )

    print(f"Done. Wrote {num_written} modified examples to {output_tfrecord_path}")

I0000 00:00:1776237056.626537    3909 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
from glob import glob
npz_dirs = ["/zfsauton/scratch/mineuih/waymax_rs/change_lane_left_es_e100_results_training/20260415_010412/traj",
            "/zfsauton/scratch/mineuih/waymax_rs/change_lane_right_es_e100_results_training/20260415_010419/traj"]
npz_paths = sorted(glob(npz_dirs[0] + "/*.npz") + glob(npz_dirs[1] + "/*.npz"))

build_modified_tfrecord(
    source_tfrecord_dir="/zfsauton/datasets/womd/tf_example/training",
    npz_paths=npz_paths,
    output_tfrecord_path="/zfsauton/scratch/mineuih/waymax_rs/additional/additional_tfexample.tfrecord-00000-of-00001",
    npz_key='trajectories_world_bkt5',
    scenario_id_suffix="_modified",
)

W0000 00:00:1776237123.669775    3909 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
I0000 00:00:1776237123.713004    4096 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


[OK] wrote modified example from training_tfexample.tfrecord-00000-of-01000[0] -> training_tfexample.tfrecord-00000-of-01000.scenario_000.npz
[OK] wrote modified example from training_tfexample.tfrecord-00000-of-01000[3] -> training_tfexample.tfrecord-00000-of-01000.scenario_003.npz
[OK] wrote modified example from training_tfexample.tfrecord-00000-of-01000[3] -> training_tfexample.tfrecord-00000-of-01000.scenario_003.npz
[OK] wrote modified example from training_tfexample.tfrecord-00000-of-01000[5] -> training_tfexample.tfrecord-00000-of-01000.scenario_005.npz
[OK] wrote modified example from training_tfexample.tfrecord-00000-of-01000[5] -> training_tfexample.tfrecord-00000-of-01000.scenario_005.npz
[OK] wrote modified example from training_tfexample.tfrecord-00000-of-01000[6] -> training_tfexample.tfrecord-00000-of-01000.scenario_006.npz
[OK] wrote modified example from training_tfexample.tfrecord-00000-of-01000[7] -> training_tfexample.tfrecord-00000-of-01000.scenario_007.npz
[OK] w